# Chapter 4: Distributed Message Queue
*System Design Interview Volume 2*

## TL;DR

A distributed message queue (like Kafka) uses **topics** split into **partitions** spread across **brokers** to decouple producers and consumers. Messages are persisted as append-only **write-ahead logs** on disk for high throughput via sequential I/O. **Consumer groups** enable parallel consumption while maintaining per-partition ordering. **Replication** with in-sync replicas (ISR) provides fault tolerance, and configurable **ACK levels** let you trade durability for latency. The system supports at-most-once, at-least-once, and exactly-once delivery semantics.

## Requirements

| Type | Requirement | Detail |
|---|---|---|
| Functional | Message production | Producers send messages to named topics |
| Functional | Message consumption | Consumers pull messages in order from partitions |
| Functional | Repeated consumption | Messages retained for configurable period (e.g. 2 weeks) |
| Functional | Delivery semantics | Configurable: at-most-once, at-least-once, exactly-once |
| Non-functional | High throughput | Support log aggregation and streaming workloads |
| Non-functional | Low latency | Configurable for traditional queue use cases |
| Non-functional | Scalability | Scale producers, consumers, brokers, partitions independently |
| Non-functional | Durability | Persist on disk, replicate across nodes |

## Estimation: Throughput and Storage

In [ ]:
# Back-of-envelope for a Kafka-like message queue
messages_per_sec = 500_000  # 500K msgs/sec peak
avg_msg_size_kb = 1  # 1 KB average message
retention_days = 14

# Throughput
throughput_mb_sec = messages_per_sec * avg_msg_size_kb / 1024
throughput_gb_hr = throughput_mb_sec * 3600 / 1024
print(f"Peak throughput: {throughput_mb_sec:.0f} MB/s")
print(f"Peak throughput: {throughput_gb_hr:.1f} GB/hr")

# Storage for retention
daily_data_tb = throughput_mb_sec * 86400 / 1024 / 1024
total_storage_tb = daily_data_tb * retention_days
replication_factor = 3
replicated_storage_tb = total_storage_tb * replication_factor
print(f"\nDaily ingestion: {daily_data_tb:.1f} TB")
print(f"14-day retention: {total_storage_tb:.0f} TB")
print(f"With 3x replication: {replicated_storage_tb:.0f} TB")

# Partitions needed
target_partition_throughput_mb = 50  # 50 MB/s per partition
min_partitions = throughput_mb_sec / target_partition_throughput_mb
print(f"\nMin partitions for throughput: {min_partitions:.0f}")

## High-Level Design

```mermaid
graph TB
    subgraph Producers
        P1[Producer 1]
        P2[Producer 2]
    end

    subgraph Coordination["Coordination (ZooKeeper/etcd)"]
        META[Metadata Storage]
        STATE[State Storage]
        COORD[Leader Election]
    end

    subgraph Brokers
        B1[Broker 1]
        B2[Broker 2]
        B3[Broker 3]
    end

    subgraph CG1["Consumer Group 1"]
        C1[Consumer 1]
        C2[Consumer 2]
    end

    subgraph CG2["Consumer Group 2"]
        C3[Consumer 3]
    end

    P1 --> B1
    P2 --> B2
    B1 <--> META
    B2 <--> META
    B3 <--> META
    B1 --> C1
    B2 --> C2
    B3 --> C3
    COORD --> B1
```

## Deep Dive

### Topics, Partitions, and Brokers

A **topic** is a named category of messages. Each topic is divided into **partitions**, which are the unit of parallelism and ordering. Partitions are distributed across broker nodes.

- Messages within a partition maintain strict **FIFO ordering** via monotonically increasing offsets
- A message key determines partition assignment: `hash(key) % num_partitions`
- If no key is provided, round-robin assignment across partitions
- Scaling a topic = adding more partitions (and brokers)

```mermaid
graph LR
    subgraph TopicA["Topic A"]
        P0["Partition 0<br/>offset: 0,1,2,..."]
        P1["Partition 1<br/>offset: 0,1,2,..."]
        P2["Partition 2<br/>offset: 0,1,2,..."]
    end

    subgraph BrokerCluster["Brokers"]
        B1["Broker 1<br/>hosts P0"]
        B2["Broker 2<br/>hosts P1"]
        B3["Broker 3<br/>hosts P2"]
    end
```

### Write-Ahead Log (WAL) Storage

Messages are persisted as **append-only segment files** on disk:

1. New messages append to the **active segment** of a partition
2. When a segment reaches a size threshold, it becomes **inactive** (read-only)
3. Old segments are **truncated** when they exceed retention
4. Sequential disk I/O on RAID configurations achieves hundreds of MB/s
5. The OS page cache aggressively caches segment data in memory

### Consumer Groups and Rebalancing

**Key constraint**: each partition is consumed by exactly one consumer within a group.

- If consumers < partitions: some consumers handle multiple partitions
- If consumers > partitions: excess consumers sit idle
- **Rebalancing** triggers when consumers join, leave, or crash
- Uses heartbeat + JoinGroup/SyncGroup protocol via a coordinator
- A leader consumer generates the partition dispatch plan

### Replication and In-Sync Replicas

Each partition has a **leader replica** and multiple **follower replicas**:

- Producers write only to the leader
- Followers continuously pull from the leader
- **ISR** = set of replicas caught up with the leader
- A follower is removed from ISR if it falls behind by more than `replica.lag.max.messages`

```mermaid
graph LR
    Producer -->|write| Leader["Leader Replica<br/>Broker 1"]
    Leader -->|fetch| F1["Follower 1<br/>Broker 2"]
    Leader -->|fetch| F2["Follower 2<br/>Broker 3"]
    Leader -->|ACK| Producer

    style Leader fill:#4a9,stroke:#333
    style F1 fill:#69b,stroke:#333
    style F2 fill:#69b,stroke:#333
```

**ACK levels control the durability-latency trade-off:**

| ACK Setting | Behavior | Durability | Latency |
|---|---|---|---|
| ACK=0 | Fire and forget, no acknowledgment | Lowest -- may lose messages | Lowest |
| ACK=1 | Leader persists before ACK | Medium -- leader failure loses data | Medium |
| ACK=all | All ISRs persist before ACK | Highest -- survives any single failure | Highest |

### Delivery Semantics

| Semantic | Producer Config | Consumer Behavior | Use Case |
|---|---|---|---|
| At-most once | ACK=0, no retry | Commit offset before processing | Monitoring metrics |
| At-least once | ACK=1 or all, with retry | Commit offset after processing | Log aggregation, most apps |
| Exactly once | Idempotent producer + transactions | Transactional offset commits | Payments, financial systems |

### Batching and Performance

Batching is applied at **three levels**:

1. **Producer**: buffer messages in memory, send in bulk (trade latency for throughput)
2. **Broker**: write batches of messages to segment files (large sequential writes)
3. **Consumer**: fetch multiple messages per pull request

In [ ]:
# Impact of batch size on effective throughput
import math

msg_size_bytes = 1024  # 1 KB
network_rtt_ms = 2
disk_write_ms = 0.5

for batch_size in [1, 10, 100, 1000, 10_000]:
    # Per-request overhead is constant; more messages per request = higher throughput
    overhead_per_request_ms = network_rtt_ms + disk_write_ms
    msgs_per_sec = batch_size / (overhead_per_request_ms / 1000)
    throughput_mb = msgs_per_sec * msg_size_bytes / (1024 * 1024)
    latency_ms = overhead_per_request_ms + (batch_size * 0.001)  # accumulation delay
    print(f"Batch={batch_size:>6}  |  {msgs_per_sec:>12,.0f} msgs/s  |  "
          f"{throughput_mb:>8.1f} MB/s  |  latency ~{latency_ms:.1f} ms")

## Trade-offs

| Dimension | Option A | Option B |
|---|---|---|
| Throughput vs Latency | Large batch size = high throughput | Small batch size = low latency |
| Durability vs Performance | ACK=all = strongest guarantee | ACK=0 = fastest writes |
| Partitions (many) | Higher parallelism, more throughput | More metadata overhead, rebalance cost |
| Replication factor (high) | Better fault tolerance | More disk and network usage |
| Pull vs Push consumers | Consumer-controlled rate, batch-friendly | Lower latency but risk of overwhelming |

## Key Takeaways

1. **Partitioning** is the key to horizontal scalability -- each partition is an independent ordered log
2. **Append-only WAL** on disk with sequential I/O outperforms random-access databases for this workload
3. **Consumer groups** elegantly support both pub-sub and point-to-point models
4. **ISR + configurable ACK** lets you dial the durability-latency knob per use case
5. **Batching everywhere** (producer, broker, consumer) is critical for high throughput
6. **ZooKeeper/etcd** handles coordination: leader election, metadata, consumer state

## Related Concepts

- [[topics-partitions-brokers]] -- the fundamental data organization model
- [[consumer-groups]] -- parallel consumption with ordering guarantees
- [[write-ahead-log]] -- append-only disk storage for high throughput
- [[replication-isr]] -- fault tolerance via leader-follower replication
- [[consumer-rebalancing]] -- dynamic partition reassignment
- [[delivery-semantics]] -- at-most/at-least/exactly-once guarantees
- [[batching-and-throughput]] -- performance optimization through batching